In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/Mujtaba7734/Advanced-Topics-in-Machine-Learning.git

%cd /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2

!git status -sb
!git log -1 --oneline

In [ ]:
%pip install -q -r requirements.txt

In [1]:
import torch
import transformers
import tokenizers
import peft
import trl
import accelerate
import datasets
import huggingface_hub

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("accelerate:", accelerate.__version__)
print("datasets:", datasets.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("CUDA:", torch.cuda.is_available())

torch: 2.10.0+cu128
transformers: 4.57.1
tokenizers: 0.22.1
peft: 0.17.1
trl: 0.27.2
accelerate: 1.13.0
datasets: 4.8.5
huggingface_hub: 0.36.2
CUDA: True


In [2]:
%cd /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2

!python -m scripts.download_assets

/kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2
Source: AbDu11aHHH/ATML-PA2-assets
Revision: 0b350481fb03f5525a35bcdec4131bd4fe487f98
added_tokens.json: 100%|███████████████████████| 605/605 [00:00<00:00, 2.28MB/s]

README.md: 5.22kB [00:00, 19.1MB/s]

adapter_config.json: 100%|█████████████████████| 860/860 [00:00<00:00, 7.86MB/s]

grpo_midpoint_metadata.json: 100%|█████████████| 308/308 [00:00<00:00, 1.54MB/s]

checkpoints/grpo_midpoint_policy/adapter(…):   0%|  | 0.00/4.37M [00:00<?, ?B/s]

cached/ppo_rollout.pt:   0%|                         | 0.00/176k [00:00<?, ?B/s]


cached/ppo_rollout_salvage_raw.pt:   0%|             | 0.00/329k [00:00<?, ?B/s]



grpo_k_cache.jsonl: 0.00B [00:00, ?B/s]




grpo_k_cache.jsonl: 360kB [00:00, 17.1MB/s]:   0%|  | 0.00/11.4M [00:00<?, ?B/s]
Fetching 90 files:   1%|▎                        | 1/90 [00:00<00:28,  3.16it/s]



chat_template.jinja: 2.51kB [00:00, 10.7MB/s]A




merges.txt: 0.00B [00:00, ?B/s]





special_tokens_map.j

In [3]:
!python -m scripts.validate_assets

dpo_standard_train           OK  data/dpo_standard_train.jsonl
dpo_standard_eval            OK  data/dpo_standard_eval.jsonl
dpo_length_train             OK  data/dpo_length_balanced_train.jsonl
dpo_length_eval              OK  data/dpo_length_stratified_eval.jsonl
word_limit_prompts           OK  data/word_limit_prompts.jsonl
rl_prompt_train              OK  data/rl_prompt_pool_train.jsonl
rl_prompt_eval               OK  data/rl_prompt_pool_eval.jsonl
xstest                       OK  data/xstest_safety_prompts.csv
gsm_train                    OK  data/gsm8k_rl_train.jsonl
gsm_eval                     OK  data/gsm8k_eval.jsonl
math_transfer_eval           OK  data/math_transfer_eval.jsonl
task5_diagnostics            OK  data/task5_controlled_reward_diagnostics.jsonl
ppo_rollout_cache            OK  cached/ppo_rollout.pt
grpo_k_cache                 OK  cached/grpo_k_cache.jsonl
ppo_midpoint_policy          OK  checkpoints/ppo_midpoint_policy
ppo_midpoint_value           OK  checkpoin

In [4]:
!python -m scripts.check_environment

Python: 3.12.13
PyTorch: 2.10.0+cu128
Transformers: 4.57.1
Tokenizers: 0.22.1
PEFT: 0.17.1
TRL: 0.27.2
CUDA available: True
GPU: Tesla T4
VRAM GiB: 14.56


In [5]:
from common.data import load_yaml
from common.models import load_tokenizer, load_policy, clear_gpu
from common.generation import batch_generate
import torch

cfg = load_yaml("configs/dpo.yaml")

tokenizer = load_tokenizer(cfg["base_model"])
model = load_policy(
    cfg,
    trainable=True,
    fresh_lora=True
)

print("Model loaded:", cfg["base_model"])
print("Device:", next(model.parameters()).device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print("Trainable parameters:", trainable)
print("Total parameters:", total)
print("Trainable %:", 100 * trainable / total)

prompt = [[
    {"role": "user", "content": "Answer in one sentence: what is machine learning?"}
]]

result = batch_generate(
    model,
    tokenizer,
    prompt,
    max_prompt_length=128,
    max_new_tokens=32,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)

print("\nGenerated response:")
print(result["responses"][0])

print("\nResponse tokens:", result["response_lengths"][0])
print("Terminated with EOS:", result["terminated_with_eos"][0])

del model
clear_gpu()

print("\nGPU memory after cleanup:")
print(round(torch.cuda.memory_allocated() / 2**30, 3), "GiB")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-1.5B-Instruct
Device: cuda:0
Trainable parameters: 1089536
Total parameters: 1544803840
Trainable %: 0.07052908413277896

Generated response:
Machine learning is a branch of artificial intelligence that involves developing algorithms and models for computer systems to automatically learn from data and improve performance on specific tasks without being explicitly programmed

Response tokens: 32
Terminated with EOS: False

GPU memory after cleanup:
0.009 GiB


In [6]:
from common.data import load_yaml
from common.models import load_reward_model, clear_gpu
from common.generation import score_reward_pairs
import torch

cfg = load_yaml("configs/dpo.yaml")

reward_model, reward_tokenizer = load_reward_model(cfg)

print("Reward model loaded:", cfg["reward_model"])
print("Device:", next(reward_model.parameters()).device)

prompt = [[
    {"role": "user", "content": "What is the capital of France?"}
]]

responses = [
    "The capital of France is Paris.",
    "The capital of France is Tokyo."
]

scores = score_reward_pairs(
    reward_model,
    reward_tokenizer,
    [prompt[0], prompt[0]],
    responses
)

print("\nReward scores:")
for response, score in zip(responses, scores):
    print(f"{float(score):.4f} -> {response}")

del reward_model
clear_gpu()

print("\nGPU memory after cleanup:")
print(round(torch.cuda.memory_allocated() / 2**30, 3), "GiB")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Reward model loaded: yavuz-ai/qwen2.5-1.5b-rm-ultrafeedback
Device: cuda:0

Reward scores:
2.3301 -> The capital of France is Paris.
-2.8457 -> The capital of France is Tokyo.

GPU memory after cleanup:
0.009 GiB


In [7]:
import torch

from task2_ppo.continue_train import prepare_ppo_continuation
from common.data import prompt_messages
from common.generation import batch_generate, response_token_logprobs
from common.models import reference_mode, token_values, clear_gpu

bundle = prepare_ppo_continuation("configs/ppo.yaml")

cfg = bundle["cfg"]
tokenizer = bundle["tokenizer"]
policy = bundle["policy"]
value_model = bundle["value_model"]

print("PPO bundle loaded")
print("Policy device:", next(policy.parameters()).device)
print("Value device:", next(value_model.parameters()).device)

policy_trainable = sum(
    p.numel() for p in policy.parameters() if p.requires_grad
)

value_trainable = sum(
    p.numel() for p in value_model.parameters() if p.requires_grad
)

print("Policy trainable params:", policy_trainable)
print("Value trainable params:", value_trainable)

# Use one actual fixed course prompt.
messages = [prompt_messages(bundle["prompt_rows"][0])]

gen = batch_generate(
    policy,
    tokenizer,
    messages,
    max_prompt_length=128,
    max_new_tokens=16,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)

seq = gen["sequences"]
attn = gen["attention_mask"]
response_ids = gen["response_ids"]
width = gen["prompt_width"]
mask = gen["response_mask"]

print("\nGenerated response:")
print(gen["responses"][0])

# Current PPO policy log-probabilities.
policy.eval()
with torch.no_grad():
    old_logp, _ = response_token_logprobs(
        policy,
        seq,
        attn,
        width,
        response_ids
    )

    # Frozen reference = same base model with PPO LoRA disabled.
    with reference_mode(policy):
        ref_logp, _ = response_token_logprobs(
            policy,
            seq,
            attn,
            width,
            response_ids
        )

print("\nold_logp shape:", tuple(old_logp.shape))
print("ref_logp shape:", tuple(ref_logp.shape))
print("response mask shape:", tuple(mask.shape))

# Test critic token-value extraction.
value_model.train()

values = token_values(
    value_model,
    seq,
    attn
)[:, width - 1:-1]

print("value shape:", tuple(values.shape))

assert old_logp.shape == ref_logp.shape
assert values.shape == old_logp.shape
assert mask.shape == old_logp.shape

print("\nTensor alignment: PASS")

# Critical test: make sure token_values() actually preserves
# gradients into the trainable critic parameters.
value_model.zero_grad(set_to_none=True)

test_loss = (values * mask).sum() / mask.sum().clamp_min(1)
test_loss.backward()

grad_params = [
    name
    for name, p in value_model.named_parameters()
    if p.requires_grad and p.grad is not None
]

finite_grads = all(
    torch.isfinite(p.grad).all().item()
    for p in value_model.parameters()
    if p.requires_grad and p.grad is not None
)

print("Critic parameters receiving gradients:", len(grad_params))
print("Critic gradients finite:", finite_grads)

print("\nExample gradient parameters:")
for name in grad_params[:8]:
    print(" ", name)

assert len(grad_params) > 0
assert finite_grads

print("\nPPO policy/value smoke test: PASS")

del policy
del value_model
del bundle
clear_gpu()

print(
    "GPU memory after cleanup:",
    round(torch.cuda.memory_allocated() / 2**30, 3),
    "GiB"
)

PPO bundle loaded
Policy device: cuda:0
Value device: cuda:0
Policy trainable params: 1089536
Value trainable params: 541568

Generated response:
To use Rclone to mount Google Drive and stream 4K files at least

old_logp shape: (1, 16)
ref_logp shape: (1, 16)
response mask shape: (1, 16)
value shape: (1, 16)

Tensor alignment: PASS
Critic parameters receiving gradients: 97
Critic gradients finite: True

Example gradient parameters:
  base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight
  base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight
  base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight
  base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight
  base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight
  base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight
  base_model.model.model.layers.1.self_attn.v_proj.lora_A.default.weight
  base_model.model.model.layers.1.self_attn.v_proj.lo

In [8]:
%cd /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2

!python -m task1_dpo.train \
    --config configs/dpo.yaml \
    --run-name smoke \
    --output outputs/task1_dpo/smoke \
    --max-examples 4

/kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [9]:
%env TOKENIZERS_PARALLELISM=false

env: TOKENIZERS_PARALLELISM=false


In [10]:
!cat results/task1_dpo/smoke_train.json

{
  "run_name": "smoke",
  "adapter": "/kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2/outputs/task1_dpo/smoke",
  "examples": 4,
  "epochs": 1,
  "optimizer_steps": 1,
  "beta": 0.1,
  "wall_seconds": 4.337313709000227
}

In [12]:
!python -m task1_dpo.train \
    --config configs/dpo.yaml \
    --run-name standard \
    --output outputs/task1_dpo/standard

In [16]:
%cd /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2

!git pull origin main

%env TOKENIZERS_PARALLELISM=false

!python -m task1_dpo.run_all \
    --config configs/dpo.yaml \
    --skip-standard-train

/kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 14 (delta 10), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (14/14), 2.46 KiB | 503.00 KiB/s, done.
From https://github.com/Mujtaba7734/Advanced-Topics-in-Machine-Learning
 * branch            main       -> FETCH_HEAD
   92332d2..22b1db9  main       -> origin/main
Updating 92332d2..22b1db9
^C
env: TOKENIZERS_PARALLELISM=false
/usr/bin/python3: No module named task1_dpo.run_all


In [17]:
%cd /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2

!git status -sb
!git pull --ff-only origin main
!git log -1 --oneline
!ls task1_dpo/run_all.py

/kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2
## main...origin/main [behind 3]
?? results/
From https://github.com/Mujtaba7734/Advanced-Topics-in-Machine-Learning
 * branch            main       -> FETCH_HEAD
Updating 92332d2..22b1db9
Fast-forward
 Assignment-2/README.md            | 14 +++++++
 Assignment-2/task1_dpo/run_all.py | 83 +++++++++++++++++++++++++++++++++++++++
 Assignment-2/task1_dpo/train.py   |  5 ++-
 3 files changed, 101 insertions(+), 1 deletion(-)
 create mode 100644 Assignment-2/task1_dpo/run_all.py
22b1db9 (HEAD -> main, origin/main, origin/HEAD) Document one-command Task 1 workflow
task1_dpo/run_all.py


In [18]:
%env TOKENIZERS_PARALLELISM=false

!python -m task1_dpo.run_all \
    --config configs/dpo.yaml \
    --skip-standard-train

env: TOKENIZERS_PARALLELISM=false
Reusing completed standard DPO adapter: /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2/outputs/task1_dpo/standard

Running: /usr/bin/python3 -m task1_dpo.evaluate --config configs/dpo.yaml --adapter /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2/outputs/task1_dpo/standard --name standard
{'n': 300, 'preference_accuracy': 0.6266666666666667, 'reference_adjusted_accuracy': 0.6266666666666667, 'policy_only_preference_accuracy': 0.49666666666666665, 'heldout_dpo_loss': 0.6763478903969129, 'beta': 0.1, 'mean_reference_log_ratio_chosen': -0.6032248520261297, 'mean_reward_margin': 0.5163464864095052, 'mean_response_reward': 1.1693970743815105, 'sampled_response_kl': 0.00019742385950495813, 'mean_response_tokens': 155.2, 'response_tokens_std': 104.4076625540482, 'mean_response_words': 110.44666666666667}

Running: /usr/bin/python3 -m task1_dpo.ablate_beta --config configs/dpo.yaml
DPO beta_0.03: 100%|█| 300/300 [09:45<00:00,

In [21]:
%cd /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2

!git pull --ff-only origin main

/kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 20 (delta 15), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (20/20), 4.20 KiB | 859.00 KiB/s, done.
From https://github.com/Mujtaba7734/Advanced-Topics-in-Machine-Learning
 * branch            main       -> FETCH_HEAD
   22b1db9..d624da9  main       -> origin/main
Updating 22b1db9..d624da9
Fast-forward
 Assignment-2/task1_dpo/ablate_beta.py    |  62 ++++++++++++--
 Assignment-2/task1_dpo/analyze_length.py | 142 ++++++++++++++++++++++++-------
 Assignment-2/task1_dpo/evaluate.py       |   3 +-
 Assignment-2/task1_dpo/run_all.py        |  75 ++++++++++------
 4 files changed, 217 insertions(+), 65 deletions(-)


In [23]:
%env TOKENIZERS_PARALLELISM=false

!python -m task1_dpo.run_all \
    --config configs/dpo.yaml \
    --resume

env: TOKENIZERS_PARALLELISM=false
[resume] Reusing completed standard DPO adapter: /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2/outputs/task1_dpo/standard
[resume] Reusing completed standard evaluation

Running: /usr/bin/python3 -m task1_dpo.ablate_beta --config configs/dpo.yaml --resume
[resume] Reusing completed training: beta_0.03
[resume] Reusing completed evaluation: beta_0.03
[resume] Reusing completed training: beta_0.10
[resume] Reusing completed evaluation: beta_0.10
[resume] Reusing completed training: beta_0.30
DPO evaluation: 100%|████████████████████| 300/300 [47:39<00:00,  9.53s/example]

Running: /usr/bin/python3 -m task1_dpo.analyze_length --config configs/dpo.yaml --resume
DPO length_balanced: 100%|█| 750/750 [19:33<00:00,  1.56s/batch, grad=3.398, los
DPO evaluation: 100%|████████████████████| 246/246 [35:47<00:00,  8.73s/example]

Task 1 pipeline complete.
Results: /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2/results/task1_dpo


SyntaxError: invalid decimal literal (1600501758.py, line 2)

In [1]:
import json
from pathlib import Path

root = Path("results/task1_dpo")

for name in [
    "standard_eval.json",
    "beta_study.json",
    "length_study.json",
]:
    path = root / name
    print("\n" + "=" * 90)
    print(name)
    print("=" * 90)
    print(json.dumps(json.loads(path.read_text()), indent=2))


standard_eval.json


FileNotFoundError: [Errno 2] No such file or directory: 'results/task1_dpo/standard_eval.json'

In [3]:
from pathlib import Path

repo = Path("/kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2")

print("Current directory:", Path.cwd())
print("Repo exists:", repo.exists())
print("Task 1 results folder exists:", (repo / "results/task1_dpo").exists())
print("Standard result exists:", (repo / "results/task1_dpo/standard_eval.json").exists())

if (repo / "results/task1_dpo").exists():
    print("\nTask 1 files:")
    for p in sorted((repo / "results/task1_dpo").iterdir()):
        print(p.name)

Current directory: /kaggle/working
Repo exists: False
Task 1 results folder exists: False
Standard result exists: False


In [4]:
%cd /kaggle/working

!git clone https://github.com/Mujtaba7734/Advanced-Topics-in-Machine-Learning.git

/kaggle/working
Cloning into 'Advanced-Topics-in-Machine-Learning'...
remote: Enumerating objects: 1056, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (144/144), done.
remote: Total 1056 (delta 82), reused 86 (delta 23), pack-reused 887 (from 3)
Receiving objects: 100% (1056/1056), 217.74 MiB | 41.46 MiB/s, done.
Resolving deltas: 100% (121/121), done.


In [5]:
%cd /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2

/kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2


In [6]:
%pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 99.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 67.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.9/530.9 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 44.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [7]:
%cd /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2

/kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2


In [8]:
!python -m scripts.download_assets
!python -m scripts.validate_assets

Source: AbDu11aHHH/ATML-PA2-assets
Revision: 0b350481fb03f5525a35bcdec4131bd4fe487f98
added_tokens.json: 100%|███████████████████████| 605/605 [00:00<00:00, 4.22MB/s]

README.md: 5.22kB [00:00, 21.9MB/s]

checkpoints/grpo_midpoint_policy/adapter(…):   0%|  | 0.00/4.37M [00:00<?, ?B/s]

cached/ppo_rollout_salvage_raw.pt:   0%|             | 0.00/329k [00:00<?, ?B/s]


cached/ppo_rollout.pt:   0%|                         | 0.00/176k [00:00<?, ?B/s]



grpo_midpoint_metadata.json: 100%|█████████████| 308/308 [00:00<00:00, 2.36MB/s]




adapter_config.json: 100%|█████████████████████| 860/860 [00:00<00:00, 9.78MB/s]




grpo_k_cache.jsonl: 0.00B [00:00, ?B/s]




chat_template.jinja: 2.51kB [00:00, 10.7MB/s]A
grpo_k_cache.jsonl: 360kB [00:00, 21.3MB/s]
Fetching 90 files:   1%|▎                        | 1/90 [00:00<00:29,  3.06it/s]



special_tokens_map.json: 100%|█████████████████| 613/613 [00:00<00:00, 3.52MB/s]




merges.txt: 0.00B [00:00, ?B/s]




merges.txt: 1.67MB [00:00, 35.2MB/s]

In [9]:
%env TOKENIZERS_PARALLELISM=false

!python -m task1_dpo.train \
    --config configs/dpo.yaml \
    --run-name smoke \
    --output outputs/task1_dpo/smoke \
    --max-examples 4

env: TOKENIZERS_PARALLELISM=false
tokenizer_config.json: 7.30kB [00:00, 35.3MB/s]
vocab.json: 2.78MB [00:00, 44.4MB/s]
merges.txt: 1.67MB [00:00, 119MB/s]
tokenizer.json: 7.03MB [00:00, 173MB/s]
config.json: 100%|█████████████████████████████| 660/660 [00:00<00:00, 4.93MB/s]
model.safetensors: 100%|████████████████████| 3.09G/3.09G [00:10<00:00, 296MB/s]
generation_config.json: 100%|██████████████████| 242/242 [00:00<00:00, 2.49MB/s]
DPO smoke: 100%|█| 2/2 [00:04<00:00,  2.26s/batch, grad=3.183, loss=0.6931, step


In [10]:
!python -m task1_dpo.evaluate \
    --config configs/dpo.yaml \
    --adapter outputs/task1_dpo/smoke \
    --name smoke \
    --max-examples 8

config.json: 1.47kB [00:00, 6.78MB/s]
model.safetensors: 100%|████████████████████| 3.09G/3.09G [00:24<00:00, 129MB/s]
DPO evaluation: 100%|████████████████████████| 8/8 [00:35<00:00,  4.45s/example]
{'n': 8, 'preference_accuracy': 0.625, 'reference_adjusted_accuracy': 0.625, 'policy_only_preference_accuracy': 0.375, 'heldout_dpo_loss': 0.6907011941075325, 'beta': 0.1, 'mean_reference_log_ratio_chosen': 0.06264877319335938, 'mean_reward_margin': 1.593414306640625, 'mean_response_reward': 0.8258056640625, 'sampled_response_kl': -4.646432191147949e-05, 'mean_response_tokens': 178.0, 'response_tokens_std': 101.86756107809786, 'mean_response_words': 121.75, 'eval_batch_size': 4, 'generation_seed_scheme': 'seed + batch_start_index'}


In [11]:
%cd /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2

!git pull --ff-only origin main

/kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 5 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 2.76 KiB | 2.76 MiB/s, done.
From https://github.com/Mujtaba7734/Advanced-Topics-in-Machine-Learning
 * branch            main       -> FETCH_HEAD
   24a3144..47a41bd  main       -> origin/main
Updating 24a3144..47a41bd
Fast-forward
 Assignment-2/task1_dpo/run_all.py | 109 ++++++++++++++++++++++++++++++++++++++
 1 file changed, 109 insertions(+)


In [12]:
%env TOKENIZERS_PARALLELISM=false

!python -m task1_dpo.run_all \
    --config configs/dpo.yaml

env: TOKENIZERS_PARALLELISM=false

Running: /usr/bin/python3 -m task1_dpo.train --config configs/dpo.yaml --run-name standard --output /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2/outputs/task1_dpo/standard
DPO standard: 100%|█| 750/750 [23:03<00:00,  1.84s/batch, grad=2.398, loss=0.596

Running: /usr/bin/python3 -m task1_dpo.evaluate --config configs/dpo.yaml --adapter /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2/outputs/task1_dpo/standard --name standard
DPO evaluation: 100%|████████████████████| 300/300 [22:37<00:00,  4.53s/example]
{'n': 300, 'preference_accuracy': 0.6333333333333333, 'reference_adjusted_accuracy': 0.6333333333333333, 'policy_only_preference_accuracy': 0.5, 'heldout_dpo_loss': 0.6769575242201488, 'beta': 0.1, 'mean_reference_log_ratio_chosen': -0.5122156963280091, 'mean_reward_margin': 0.5189518737792969, 'mean_response_reward': 1.1548957379659017, 'sampled_response_kl': 0.0005818313363316244, 'mean_response_tokens': 157.6733

In [1]:
!tar -czf /kaggle/working/ATML_PA2_Task1_EMERGENCY_BACKUP.tar.gz \
    -C /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2 \
    results/task1_dpo outputs/task1_dpo

In [2]:
!ls -lh /kaggle/working/ATML_PA2_Task1_EMERGENCY_BACKUP.tar.gz

-rw-r--r-- 1 root root 23M Oct  1 16:11 /kaggle/working/ATML_PA2_Task1_EMERGENCY_BACKUP.tar.gz


In [3]:
%cd /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2

!git pull --ff-only origin main

/kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 2.08 KiB | 1.04 MiB/s, done.
From https://github.com/Mujtaba7734/Advanced-Topics-in-Machine-Learning
 * branch            main       -> FETCH_HEAD
   47a41bd..6d1713a  main       -> origin/main
Updating 47a41bd..6d1713a
Fast-forward
 Assignment-2/task1_dpo/run_all.py | 117 +++++++++++++++++++++++++++++++++++++-
 1 file changed, 116 insertions(+), 1 deletion(-)


In [4]:
from common.data import load_yaml
from task1_dpo.run_all import validate_task1_artifacts, create_verified_backup

cfg = load_yaml("configs/dpo.yaml")

validate_task1_artifacts(cfg)
backup = create_verified_backup(cfg)

print("\nFINAL VERIFIED BACKUP:", backup)


TASK 1 COMPLETENESS CHECK: PASS
  Standard train:       1500/1500 examples, 1 epoch
  Standard eval:        300/300 pairs
  Beta short runs:      3 x 600 examples
  Beta evals:           3 x 300 pairs
  Length-balanced train:1500/1500 pairs, 1 epoch
  Stratified evals:     2 x 246 pairs
  Word-limit evals:     2 x 10 prompts
  No Task 1 experimental budget was shortened.

TASK 1 BACKUP CREATED AND VERIFIED
Backup:   /kaggle/working/ATML_PA2_Task1_BACKUP.tar.gz
SHA256:   45f9a9fb844d31326e9a45d2d56e58e1b703fd817d1b4570abda4fe3296f2ee9
Checksum: /kaggle/working/ATML_PA2_Task1_BACKUP.tar.gz.sha256
Size MiB: 20.0
IMPORTANT: preserve this /kaggle/working backup with Kaggle Save Version/output
or download it before stopping/resetting the session.

FINAL VERIFIED BACKUP: /kaggle/working/ATML_PA2_Task1_BACKUP.tar.gz


In [5]:
%cd /kaggle/working

!sha256sum -c ATML_PA2_Task1_BACKUP.tar.gz.sha256

/kaggle/working
ATML_PA2_Task1_BACKUP.tar.gz: OK


In [6]:
%cd /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2

!git pull --ff-only origin main

/kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 29 (delta 22), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (29/29), 8.26 KiB | 939.00 KiB/s, done.
From https://github.com/Mujtaba7734/Advanced-Topics-in-Machine-Learning
 * branch            main       -> FETCH_HEAD
   6d1713a..ee73b94  main       -> origin/main
Updating 6d1713a..ee73b94
Fast-forward
 Assignment-2/README.md                     |  14 ++
 Assignment-2/common/rl_evaluation.py       |   3 +-
 Assignment-2/task2_ppo/ablate_kl.py        |  80 ++++++-
 Assignment-2/task2_ppo/analyze_clipping.py | 102 ++++++++-
 Assignment-2/task2_ppo/continue_train.py   |  10 +-
 Assignment-2/task2_ppo/run_all.py          | 337 +++++++++++++++++++++++++++++
 6 files changed, 525 insertions(+), 21 deletions(-)
 create mode 100644 Assignment-2/task2_ppo/run_all.

In [13]:
!python -m pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [15]:
import importlib.util

print("torchao:", importlib.util.find_spec("torchao"))

torchao: None


In [17]:
!python -m py_compile \
    task2_ppo/run_all.py \
    task2_ppo/continue_train.py \
    task2_ppo/analyze_clipping.py \
    task2_ppo/ablate_kl.py \
    common/rl_evaluation.py

In [19]:
!python -m pip install -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.2 MB/s eta 0:00:00:00:0100:01


In [20]:
import bitsandbytes as bnb
print(bnb.__version__)

0.50.2


In [21]:
%env TOKENIZERS_PARALLELISM=false

!python -m task2_ppo.continue_train \
    --config configs/ppo.yaml \
    --run-name smoke \
    --output outputs/task2_ppo/smoke \
    --updates 1

env: TOKENIZERS_PARALLELISM=false
Loading weights: 100%|█| 338/338 [00:01<00:00, 237.01it/s, Materializing param=m
Loading weights: 100%|█| 291/291 [00:00<00:00, 559.78it/s, Materializing param=s
model.safetensors: 100%|████████████████████| 3.09G/3.09G [00:10<00:00, 308MB/s]
Loading weights: 100%|█| 339/339 [00:04<00:00, 73.75it/s, Materializing param=sc
PPO smoke:   0%|                                      | 0/1 [00:00<?, ?update/s]MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat1

In [22]:
%cd /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2
!git pull --ff-only origin main

/kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 9 (delta 7), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 1.03 KiB | 352.00 KiB/s, done.
From https://github.com/Mujtaba7734/Advanced-Topics-in-Machine-Learning
 * branch            main       -> FETCH_HEAD
   ee73b94..a176fb2  main       -> origin/main
Updating ee73b94..a176fb2
Fast-forward
 Assignment-2/common/generation.py | 7 +++++++
 Assignment-2/requirements.txt     | 2 +-
 2 files changed, 8 insertions(+), 1 deletion(-)


In [23]:
!python -m py_compile common/generation.py task2_ppo/continue_train.py

In [24]:
%env TOKENIZERS_PARALLELISM=false

!python -m task2_ppo.continue_train \
    --config configs/ppo.yaml \
    --run-name smoke \
    --output outputs/task2_ppo/smoke \
    --updates 1

env: TOKENIZERS_PARALLELISM=false
Loading weights: 100%|█| 338/338 [00:01<00:00, 240.21it/s, Materializing param=m
Loading weights: 100%|█| 291/291 [00:00<00:00, 557.09it/s, Materializing param=s
Loading weights: 100%|█| 339/339 [00:04<00:00, 73.95it/s, Materializing param=sc
PPO smoke:   0%|                                      | 0/1 [00:00<?, ?update/s]MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfl

In [25]:
%cd /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2

!rm -rf outputs/task2_ppo/smoke outputs/task2_ppo/smoke_value
!rm -f results/task2_ppo/smoke_train.json \
       results/task2_ppo/smoke_train.jsonl \
       results/task2_ppo/smoke_config.json

/kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2


In [26]:
%env TOKENIZERS_PARALLELISM=false

!python -m task2_ppo.run_all \
    --config configs/ppo.yaml

env: TOKENIZERS_PARALLELISM=false

Running: /usr/bin/python3 -m task2_ppo.continue_train --config configs/ppo.yaml --run-name standard --output /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2/outputs/task2_ppo/standard
Loading weights: 100%|█| 338/338 [00:01<00:00, 245.15it/s, Materializing param=m
Loading weights: 100%|█| 291/291 [00:00<00:00, 569.89it/s, Materializing param=s
Loading weights: 100%|█| 339/339 [00:04<00:00, 72.96it/s, Materializing param=sc
PPO standard:   0%|                                  | 0/20 [00:00<?, ?update/s]MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: in

In [27]:
%cd /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2
!git pull --ff-only origin main

/kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 970 bytes | 970.00 KiB/s, done.
From https://github.com/Mujtaba7734/Advanced-Topics-in-Machine-Learning
 * branch            main       -> FETCH_HEAD
   a176fb2..054946a  main       -> origin/main
Updating a176fb2..054946a
Fast-forward
 Assignment-2/task2_ppo/continue_train.py | 28 +++++++++++++++++++++++-----
 1 file changed, 23 insertions(+), 5 deletions(-)


In [28]:
!python -m py_compile task2_ppo/continue_train.py

In [30]:
%env TOKENIZERS_PARALLELISM=false

!python -m task2_ppo.continue_train \
    --config configs/ppo.yaml \
    --run-name smoke3 \
    --output outputs/task2_ppo/smoke3 \
    --updates 3

env: TOKENIZERS_PARALLELISM=false
Loading weights: 100%|█| 338/338 [00:01<00:00, 238.59it/s, Materializing param=m
Loading weights: 100%|█| 291/291 [00:00<00:00, 549.94it/s, Materializing param=s
Loading weights: 100%|█| 339/339 [00:04<00:00, 75.85it/s, Materializing param=sc
PPO smoke3:   0%|                                     | 0/3 [00:00<?, ?update/s]MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfl

In [1]:
%cd /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2
!git pull --ff-only origin main

/kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2
remote: Enumerating objects: 16, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 10 (delta 8), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 1.32 KiB | 449.00 KiB/s, done.
From https://github.com/Mujtaba7734/Advanced-Topics-in-Machine-Learning
 * branch            main       -> FETCH_HEAD
   054946a..45d545b  main       -> origin/main
Updating 054946a..45d545b
Fast-forward
 Assignment-2/common/models.py            | 15 +++++++++++++++
 Assignment-2/task2_ppo/continue_train.py | 16 +++++++++++-----
 2 files changed, 26 insertions(+), 5 deletions(-)


In [2]:
!python -m py_compile \
    common/models.py \
    task2_ppo/continue_train.py

In [3]:
%env TOKENIZERS_PARALLELISM=false

!python -m task2_ppo.continue_train \
    --config configs/ppo.yaml \
    --run-name smoke3 \
    --output outputs/task2_ppo/smoke3 \
    --updates 3

env: TOKENIZERS_PARALLELISM=false
config.json: 100%|█████████████████████████████| 660/660 [00:00<00:00, 2.54MB/s]
tokenizer_config.json: 7.30kB [00:00, 17.1MB/s]
vocab.json: 2.78MB [00:00, 34.7MB/s]
merges.txt: 1.67MB [00:00, 119MB/s]
tokenizer.json: 7.03MB [00:00, 152MB/s]
model.safetensors: 100%|████████████████████| 3.09G/3.09G [00:11<00:00, 270MB/s]
Loading weights: 100%|█| 338/338 [00:01<00:00, 226.78it/s, Materializing param=m
generation_config.json: 100%|██████████████████| 242/242 [00:00<00:00, 1.76MB/s]
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2/task2_ppo/continue_train.py", line 200, in <module>
    main()
  File "/kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2/task2_ppo/continue_train.py", line 196, in main
    run_ppo(args.config, args.output, args.updates, args.clip_epsilon, args.kl_beta,

In [4]:
%cd /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2
!git pull --ff-only origin main

/kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 10 (delta 7), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 3.00 KiB | 1022.00 KiB/s, done.
From https://github.com/Mujtaba7734/Advanced-Topics-in-Machine-Learning
 * branch            main       -> FETCH_HEAD
   45d545b..59535c2  main       -> origin/main
Updating 45d545b..59535c2
Fast-forward
 Assignment-2/task2_ppo/run_all.py   | 75 ++++++++++++++++++++++++++++++++++
 Assignment-2/task2_ppo/self_test.py | 80 +++++++++++++++++++++++++++++++++++++
 2 files changed, 155 insertions(+)
 create mode 100644 Assignment-2/task2_ppo/self_test.py


In [5]:
%env TOKENIZERS_PARALLELISM=false

!python -m task2_ppo.run_all \
    --config configs/ppo.yaml \
    --resume

env: TOKENIZERS_PARALLELISM=false

TASK 2 RUNTIME PRECHECK
  Removing unused torchao 0.10.0 to avoid PEFT dispatch conflicts...
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
  Repairing pinned course Python dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 89.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 56.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.9/530.9 kB 14.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 14.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 17.4 MB/s eta 0:00:00
  Attempting uninstall: hugging

In [3]:
from pathlib import Path

root = Path("/kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2")

print("Repo exists:", root.exists())

checks = {
    "standard policy": root / "outputs/task2_ppo/standard/adapter_config.json",
    "standard value": root / "outputs/task2_ppo/standard_value/adapter_config.json",
    "standard train summary": root / "results/task2_ppo/standard_train.json",
    "standard train log": root / "results/task2_ppo/standard_train.jsonl",
    "standard eval": root / "results/task2_ppo/standard_eval.json",
    "standard examples": root / "results/task2_ppo/standard_examples.jsonl",
}

for name, path in checks.items():
    print(f"{name:25s}:", path.exists())

Repo exists: True
standard policy          : True
standard value           : True
standard train summary   : True
standard train log       : True
standard eval            : True
standard examples        : True


In [ ]:
%cd /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2
!git pull --ff-only origin main

In [4]:
%cd /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2

!timeout 60 git pull --ff-only origin main
!git rev-parse --short HEAD

/kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 15 (delta 12), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (15/15), 3.12 KiB | 797.00 KiB/s, done.
From https://github.com/Mujtaba7734/Advanced-Topics-in-Machine-Learning
 * branch            main       -> FETCH_HEAD
   59535c2..acaf9ec  main       -> origin/main
Updating 59535c2..acaf9ec
Fast-forward
 Assignment-2/scripts/validate_assets.py    | 31 +++++++++++++
 Assignment-2/task2_ppo/analyze_clipping.py | 71 +++++++++++++++++++++++++++---
 Assignment-2/task2_ppo/run_all.py          | 10 +++++
 3 files changed, 105 insertions(+), 7 deletions(-)
acaf9ec


In [5]:
%env TOKENIZERS_PARALLELISM=false

!python -m task2_ppo.run_all \
    --config configs/ppo.yaml \
    --resume

env: TOKENIZERS_PARALLELISM=false

TASK 2 RUNTIME PRECHECK
  Removing unused torchao 0.10.0 to avoid PEFT dispatch conflicts...
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
  Repairing pinned course Python dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 67.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 80.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.9/530.9 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
   

In [1]:
from pathlib import Path
import hashlib

backup = Path("/kaggle/working/ATML_PA2_Task2_BACKUP.tar.gz")
sha_file = Path("/kaggle/working/ATML_PA2_Task2_BACKUP.tar.gz.sha256")

print("Backup exists:", backup.exists())
print("Checksum file exists:", sha_file.exists())

if backup.exists():
    h = hashlib.sha256()
    with backup.open("rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)

    digest = h.hexdigest()

    print("Size MiB:", round(backup.stat().st_size / 1024**2, 1))
    print("Actual SHA256:", digest)
    print(
        "MATCH:",
        digest == "5781403c40aa8a646c4cd057be279b0fc1ca28fdad5bb0c62caabd44f4bb0b3f"
    )

Backup exists: True
Checksum file exists: True
Size MiB: 41.3
Actual SHA256: 5781403c40aa8a646c4cd057be279b0fc1ca28fdad5bb0c62caabd44f4bb0b3f
MATCH: True


In [1]:
%cd /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2

!timeout 60 git pull --ff-only origin main
!git log -1 --oneline

/kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 44 (delta 28), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (44/44), 20.72 KiB | 1.38 MiB/s, done.
From https://github.com/Mujtaba7734/Advanced-Topics-in-Machine-Learning
 * branch            main       -> FETCH_HEAD
   acaf9ec..7f2c269  main       -> origin/main
Updating acaf9ec..7f2c269
Fast-forward
 Assignment-2/README.md                           |  13 +-
 Assignment-2/common/rl_evaluation.py             | 220 +++++++-
 Assignment-2/configs/grpo.yaml                   |   2 +
 Assignment-2/task3_grpo/analyze_group_size.py    | 257 +++++++--
 Assignment-2/task3_grpo/compare_normalization.py | 305 ++++++++++-
 Assignment-2/task3_grpo/continue_train.py        | 405 ++++++++++++--
 Assignment-2/task3_grpo/evaluate.py              |  14 +-
 Assignment-2/ta

In [2]:
%env TOKENIZERS_PARALLELISM=false

!python -m task3_grpo.run_all \
    --config configs/grpo.yaml \
    --resume

env: TOKENIZERS_PARALLELISM=false

TASK 3 RUNTIME PRECHECK
  Removing unused torchao 0.10.0 to avoid PEFT dispatch conflicts...
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
  Repairing pinned course Python dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 117.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 81.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.9/530.9 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 50.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
  

In [1]:
%cd /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2

!timeout 60 git pull --ff-only origin main
!git log -1 --oneline

/kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 34 (delta 21), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (34/34), 15.97 KiB | 442.00 KiB/s, done.
From https://github.com/Mujtaba7734/Advanced-Topics-in-Machine-Learning
 * branch            main       -> FETCH_HEAD
   7f2c269..d033c11  main       -> origin/main
Updating 7f2c269..d033c11
Fast-forward
 Assignment-2/README.md                          |  18 +-
 Assignment-2/configs/feedback.yaml              |   2 +
 Assignment-2/task4_safety/evaluate_safety.py    | 326 ++++++++++++---
 Assignment-2/task4_safety/generate_responses.py | 202 ++++++++--
 Assignment-2/task4_safety/judge_responses.py    | 223 +++++++++--
 Assignment-2/task4_safety/run_all.py            | 505 ++++++++++++++++++++++++
 Assignment-2/task4_safety/self_test.py          |  87 ++++
 

In [2]:
%env TOKENIZERS_PARALLELISM=false

!python -m task4_safety.run_all \
    --config configs/feedback.yaml \
    --resume

env: TOKENIZERS_PARALLELISM=false

TASK 4 RUNTIME PRECHECK
  Removing unused torchao 0.10.0 to avoid PEFT dispatch conflicts...
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
  Repairing pinned course Python dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 562.9 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 53.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 90.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.9/530.9 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.7 MB/s eta 0:00:00
  Attempting uninstall: huggingfac

In [3]:
%cd /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2

!timeout 60 git pull --ff-only origin main
!git log -1 --oneline

/kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 10 (delta 8), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 1.56 KiB | 400.00 KiB/s, done.
From https://github.com/Mujtaba7734/Advanced-Topics-in-Machine-Learning
 * branch            main       -> FETCH_HEAD
   d033c11..7906357  main       -> origin/main
Updating d033c11..7906357
Fast-forward
 Assignment-2/task4_safety/judge_responses.py | 24 +++++++++++++++++++++++-
 Assignment-2/task4_safety/run_all.py         | 11 ++++++++++-
 2 files changed, 33 insertions(+), 2 deletions(-)
7906357 (HEAD -> main, origin/main, origin/HEAD) Require the corrected Task 4 judge protocol in validation


In [4]:
%env TOKENIZERS_PARALLELISM=false

!python -m task4_safety.run_all \
    --config configs/feedback.yaml \
    --resume

env: TOKENIZERS_PARALLELISM=false

TASK 4 RUNTIME PRECHECK
  torchao:      absent (intentional; not used by PA2)
  bitsandbytes: 0.50.2
  transformers: 4.57.1
  tokenizers: 0.22.1
  peft: 0.17.1
  trl: 0.27.2
TASK 4 RUNTIME PRECHECK: PASS

Running: /usr/bin/python3 -m scripts.check_environment
Python: 3.12.13
PyTorch: 2.10.0+cu128
Transformers: 4.57.1
Tokenizers: 0.22.1
PEFT: 0.17.1
TRL: 0.27.2
CUDA available: True
GPU: Tesla T4
VRAM GiB: 14.56

Running: /usr/bin/python3 -m scripts.validate_assets
dpo_standard_train           OK  data/dpo_standard_train.jsonl
dpo_standard_eval            OK  data/dpo_standard_eval.jsonl
dpo_length_train             OK  data/dpo_length_balanced_train.jsonl
dpo_length_eval              OK  data/dpo_length_stratified_eval.jsonl
word_limit_prompts           OK  data/word_limit_prompts.jsonl
rl_prompt_train              OK  data/rl_prompt_pool_train.jsonl
rl_prompt_eval               OK  data/rl_prompt_pool_eval.jsonl
xstest                       OK  data/x

In [1]:
%cd /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2

!timeout 60 git pull --ff-only origin main
!git log -1 --oneline

/kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 44 (delta 30), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (44/44), 16.64 KiB | 631.00 KiB/s, done.
From https://github.com/Mujtaba7734/Advanced-Topics-in-Machine-Learning
 * branch            main       -> FETCH_HEAD
   7906357..9542c36  main       -> origin/main
Updating 7906357..9542c36
Fast-forward
 Assignment-2/README.md                             |   9 +-
 Assignment-2/configs/feedback.yaml                 |   3 +
 Assignment-2/task5_feedback/compare_feedback.py    |  58 ++-
 Assignment-2/task5_feedback/evaluate_math.py       | 397 +++++++++++++++---
 Assignment-2/task5_feedback/rlaif.py               |  19 +-
 Assignment-2/task5_feedback/run_all.py             | 460 +++++++++++++++++++++
 Assignment-2/task5_feedback/score_perturbations.py | 359 +

In [ ]:
%env TOKENIZERS_PARALLELISM=false

!python -m task5_feedback.run_all \
    --config configs/feedback.yaml

env: TOKENIZERS_PARALLELISM=false

TASK 5 RUNTIME PRECHECK
  Removing unused torchao 0.10.0 to avoid PEFT dispatch conflicts...
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
  Repairing pinned course Python dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 78.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 63.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.9/530.9 kB 15.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 40.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface_h

In [1]:
%cd /kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2

!timeout 60 git pull --ff-only origin main
!git log -1 --oneline

/kaggle/working/Advanced-Topics-in-Machine-Learning/Assignment-2
From https://github.com/Mujtaba7734/Advanced-Topics-in-Machine-Learning
 * branch            main       -> FETCH_HEAD
Already up to date.
9542c36 (HEAD -> main, origin/main, origin/HEAD) Document the verified Task 5 evaluation workflow


In [2]:
%env TOKENIZERS_PARALLELISM=false

!python -m task5_feedback.run_all \
    --config configs/feedback.yaml

env: TOKENIZERS_PARALLELISM=false

TASK 5 RUNTIME PRECHECK
  Removing unused torchao 0.10.0 to avoid PEFT dispatch conflicts...
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
  Repairing pinned course Python dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 57.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 36.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.9/530.9 kB 14.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 29.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 15.0 MB/s eta 0:00:0000:01
  Attempting uninstall: huggin